In [1]:
import xml.etree.ElementTree as et
import pandas as pd
from io import StringIO
import psycopg2

In [2]:
def get_data(data_file_path, et=et ):
    '''
    Gets xml data for the steps data.

    Parameters
    data_file_path: file path to the steps data

    Types:

    Returns:
    - A dataframe containing all the records of the health data
    '''
    
    file_path=data_file_path

    tree = et.parse(file_path)
    root = tree.getroot()

    data = []

    for record in root.findall('Record'):
        if record.get('type'):
            child_elements={
                'type': record.get('type'),
                'source_name': record.get('sourceName'),
                'source_version': record.get('sourceVersion'),
                'device': record.get('sourceName'),
                'unit': record.get('unit'),
                'creation_date': record.get('creationDate'),
                'start_date': record.get('startDate'),
                'end_date': record.get('endDate'),
                'value': record.get('value')
            }

            data.append(child_elements)

    df = pd.DataFrame(data)
    
    return df

In [3]:
df = get_data('oli_export.xml')

In [4]:
def get_postgres_type(dtype):
    if dtype == 'int64':
        return 'INTEGER'
    elif dtype == 'float64':
        return 'FLOAT'
    elif dtype == 'object':
        return 'TEXT'
    elif dtype == 'bool':
        return 'BOOLEAN'
    else:
        return 'TEXT'

In [5]:
table_name = 'oli'

cols_with_types = []
for col, dtype in zip(df.columns, df.dtypes):
    col_type = get_postgres_type(dtype)
    cols_with_types.append(f"{col} {col_type}")

create_table_query = f"""

CREATE TABLE IF NOT EXISTS {table_name}(
    {', '.join(cols_with_types)}
    
    );

"""

In [6]:
cols_with_types

['type TEXT',
 'source_name TEXT',
 'source_version TEXT',
 'device TEXT',
 'unit TEXT',
 'creation_date TEXT',
 'start_date TEXT',
 'end_date TEXT',
 'value TEXT']

In [7]:
dbname = 'step_count'
user = 'postgres'
password = 'NoahAnd6'
host = 'localhost' 
port = '5432'

In [8]:
conn = psycopg2.connect(
    dbname=dbname,      
    user=user,        
    password=password,    
    host=host,        
    port=port         
)

cur = conn.cursor()


In [9]:
cur.execute(create_table_query)
conn.commit()

In [10]:
df_tuples = [tuple(x) for x in df.to_numpy()]

In [11]:
cols = ', '.join(list(df.columns))
values = ', '.join(['%s' for _ in range(len(df.columns))])
insert_query = f"INSERT INTO oli ({cols}) VALUES ({values})"

In [12]:
try:
    cur.executemany(insert_query, df_tuples)

    conn.commit()

    print("Data inserted successfully")

except Exception as e:
    print(f"An error occurred: {e}")

# finally:
#     if cur:
#         cur.close()
#     if conn:
#         conn.close()

Data inserted successfully


### Processed Table

In [13]:
cols_with_types

['type TEXT',
 'source_name TEXT',
 'source_version TEXT',
 'device TEXT',
 'unit TEXT',
 'creation_date TEXT',
 'start_date TEXT',
 'end_date TEXT',
 'value TEXT']

In [18]:
processed_table_name = 'processed_oli'
cols_to_keep = 'type', 'creation_date', 'value'
cols = ', '.join(cols_to_keep)

create_processed_table_query = f"""

CREATE TABLE {processed_table_name} AS
SELECT {cols}
FROM {table_name};

"""
try:
    cur.execute(create_processed_table_query)
    conn.commit()
    print(f"Data inserted successfully in to {processed_table_name}")

except Exception as e:
    print(f"An error occurred: {e}")
    conn.rollback()


Data inserted successfully in to processed_oli


In [19]:
col_name = 'date'
col_type = 'DATE'

add_column_query = f"""

ALTER TABLE {processed_table_name} 
ADD COLUMN {col_name} {col_type};

"""

try:
    cur.execute(add_column_query)
    conn.commit()
    print(f"Column added successfully")

except Exception as e:
    print(f"An error occurred: {e}")
    conn.rollback()

Column added successfully


In [21]:
col_split_query = f"""

UPDATE {processed_table_name}
SET 
    date = SPLIT_PART(CREATION_DATE, ' ', 1)::DATE
WHERE creation_date IS NOT NULL;

"""

try:
    cur.execute(col_split_query)
    conn.commit()
    print("Column split successfully")

except Exception as e:
    print(f"An error occurred: {e}")
    conn.rollback()

Column split successfully


In [ ]:
cols_to_save = 'type'

In [22]:
cols_with_types

['type TEXT',
 'source_name TEXT',
 'source_version TEXT',
 'device TEXT',
 'unit TEXT',
 'creation_date TEXT',
 'start_date TEXT',
 'end_date TEXT',
 'value TEXT']

In [ ]:
col_drop_query = f"""

ALTER TABLE {processed_table_name}

"""